# VORq — Agent-Based Model (ABM) for Crisis Contagion

Uses Mesa to simulate heterogeneous market participants during geopolitical crises.

**Agents:**
- Institutional Investors (hedge funds, pension funds)
- Central Banks (rate setters, liquidity providers)
- Retail Traders (herd behavior, panic selling)
- Supply Chain Managers (inventory, rerouting)

**What this captures:** Emergent phenomena that top-down models miss:
- Liquidity dry-ups and flash crashes
- Panic selling cascades
- Supply chain bottleneck amplification
- Central bank intervention damping

⚡ **GPU recommended for large-scale simulations (100k+ agents)**

In [ ]:
!pip install mesa numpy matplotlib pandas

In [ ]:
import mesa
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from enum import Enum
import json

## 1. Define Agent Types

In [ ]:
class Sentiment(Enum):
    CALM = 0
    ANXIOUS = 1
    PANIC = 2


class InstitutionalInvestor(mesa.Agent):
    """Hedge fund / pension fund that sells during panic."""
    
    def __init__(self, model):
        super().__init__(model)
        self.portfolio_value = np.random.uniform(100, 1000)
        self.risk_tolerance = np.random.uniform(0.3, 0.9)
        self.sentiment = Sentiment.CALM
        self.selling = False
    
    def step(self):
        # Check neighbors' sentiment
        neighbors = self.model.grid.get_neighbors(self.pos, moore=True, include_center=False)
        panic_ratio = sum(1 for n in neighbors if hasattr(n, 'sentiment') and n.sentiment == Sentiment.PANIC) / max(len(neighbors), 1)
        
        # Update sentiment based on shock + neighbors
        shock = self.model.current_shock
        stress = shock * (1 - self.risk_tolerance) + panic_ratio * 0.4
        
        if stress > 0.7:
            self.sentiment = Sentiment.PANIC
            self.selling = True
            self.portfolio_value *= (1 - shock * 0.15)
        elif stress > 0.4:
            self.sentiment = Sentiment.ANXIOUS
            self.selling = np.random.random() < 0.3
            self.portfolio_value *= (1 - shock * 0.05)
        else:
            self.sentiment = Sentiment.CALM
            self.selling = False


class CentralBank(mesa.Agent):
    """Central bank that injects liquidity during crises."""
    
    def __init__(self, model):
        super().__init__(model)
        self.rate = 5.0
        self.intervention_threshold = 0.6
        self.intervening = False
    
    def step(self):
        if self.model.current_shock > self.intervention_threshold:
            self.intervening = True
            self.rate = max(0, self.rate - 0.25)
            # Reduce shock impact slightly
            self.model.current_shock *= 0.95
        else:
            self.intervening = False


class RetailTrader(mesa.Agent):
    """Retail traders with high herd behavior."""
    
    def __init__(self, model):
        super().__init__(model)
        self.portfolio_value = np.random.uniform(10, 100)
        self.herd_sensitivity = np.random.uniform(0.5, 1.0)
        self.sentiment = Sentiment.CALM
        self.selling = False
    
    def step(self):
        neighbors = self.model.grid.get_neighbors(self.pos, moore=True, include_center=False)
        selling_ratio = sum(1 for n in neighbors if hasattr(n, 'selling') and n.selling) / max(len(neighbors), 1)
        
        # Retail traders have high herd behavior
        if selling_ratio > 0.5 * (1 - self.herd_sensitivity):
            self.sentiment = Sentiment.PANIC
            self.selling = True
            self.portfolio_value *= (1 - self.model.current_shock * 0.20)
        elif self.model.current_shock > 0.5:
            self.sentiment = Sentiment.ANXIOUS
            self.selling = np.random.random() < 0.5
        else:
            self.sentiment = Sentiment.CALM
            self.selling = False


class SupplyChainManager(mesa.Agent):
    """Supply chain managers — inventory and rerouting."""
    
    def __init__(self, model):
        super().__init__(model)
        self.inventory_months = np.random.uniform(1, 6)
        self.bottleneck = False
    
    def step(self):
        shock = self.model.current_shock
        if shock > 0.3:
            self.inventory_months -= shock * 0.3
            if self.inventory_months < 1:
                self.bottleneck = True
        else:
            self.inventory_months = min(self.inventory_months + 0.1, 6)
            self.bottleneck = False

print('✅ Agent types defined')

## 2. Build the Crisis Simulation Model

In [ ]:
class CrisisModel(mesa.Model):
    """Agent-based model for geopolitical crisis contagion."""
    
    def __init__(
        self,
        n_institutional=200,
        n_retail=500,
        n_supply_chain=100,
        n_central_banks=5,
        initial_shock=0.3,
        shock_persistence=0.95,
        grid_size=30,
    ):
        super().__init__()
        self.grid = mesa.space.MultiGrid(grid_size, grid_size, torus=True)
        self.current_shock = initial_shock
        self.shock_persistence = shock_persistence
        self.history = []
        
        # Create agents
        for _ in range(n_institutional):
            a = InstitutionalInvestor(self)
            x, y = self.random.randrange(grid_size), self.random.randrange(grid_size)
            self.grid.place_agent(a, (x, y))
        
        for _ in range(n_retail):
            a = RetailTrader(self)
            x, y = self.random.randrange(grid_size), self.random.randrange(grid_size)
            self.grid.place_agent(a, (x, y))
        
        for _ in range(n_supply_chain):
            a = SupplyChainManager(self)
            x, y = self.random.randrange(grid_size), self.random.randrange(grid_size)
            self.grid.place_agent(a, (x, y))
        
        for _ in range(n_central_banks):
            a = CentralBank(self)
            x, y = self.random.randrange(grid_size), self.random.randrange(grid_size)
            self.grid.place_agent(a, (x, y))
    
    def step(self):
        self.agents.shuffle_do('step')
        
        # Shock decays over time
        self.current_shock *= self.shock_persistence
        
        # Add contagion amplification from panic selling
        n_panic = sum(1 for a in self.agents if hasattr(a, 'sentiment') and a.sentiment == Sentiment.PANIC)
        n_total = sum(1 for a in self.agents if hasattr(a, 'sentiment'))
        panic_ratio = n_panic / max(n_total, 1)
        
        # Panic selling feeds back into shock
        if panic_ratio > 0.3:
            self.current_shock = min(self.current_shock + panic_ratio * 0.05, 1.0)
        
        # Record state
        n_selling = sum(1 for a in self.agents if hasattr(a, 'selling') and a.selling)
        n_bottleneck = sum(1 for a in self.agents if hasattr(a, 'bottleneck') and a.bottleneck)
        total_value = sum(a.portfolio_value for a in self.agents if hasattr(a, 'portfolio_value'))
        
        self.history.append({
            'shock': self.current_shock,
            'panic_ratio': panic_ratio,
            'selling_agents': n_selling,
            'bottlenecks': n_bottleneck,
            'total_portfolio_value': total_value,
        })

print('✅ CrisisModel defined')

## 3. Run Crisis Simulations

In [ ]:
# Simulate different shock severities
shock_levels = {
    'mild': 0.2,
    'moderate': 0.5,
    'severe': 0.8,
    'extreme': 0.95,
}

results = {}
n_steps = 60  # simulate 60 time steps (approx 60 trading days)

for severity_name, shock_val in shock_levels.items():
    print(f'\n🔄 Simulating {severity_name} crisis (shock={shock_val})...')
    model = CrisisModel(initial_shock=shock_val)
    
    for _ in range(n_steps):
        model.step()
    
    results[severity_name] = model.history
    final = model.history[-1]
    print(f'  Final shock: {final["shock"]:.4f}')
    print(f'  Peak panic ratio: {max(h["panic_ratio"] for h in model.history):.2%}')
    print(f'  Final portfolio value: {final["total_portfolio_value"]:,.0f}')

print('\n✅ All simulations complete')

## 4. Visualize Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
colors = {'mild': '#34D399', 'moderate': '#FBBF24', 'severe': '#FB923C', 'extreme': '#F87171'}

for severity_name, history in results.items():
    c = colors[severity_name]
    steps = range(len(history))
    
    axes[0,0].plot(steps, [h['shock'] for h in history], label=severity_name, color=c, linewidth=2)
    axes[0,1].plot(steps, [h['panic_ratio'] for h in history], label=severity_name, color=c, linewidth=2)
    axes[1,0].plot(steps, [h['bottlenecks'] for h in history], label=severity_name, color=c, linewidth=2)
    axes[1,1].plot(steps, [h['total_portfolio_value'] for h in history], label=severity_name, color=c, linewidth=2)

axes[0,0].set_title('Shock Persistence', fontsize=14)
axes[0,1].set_title('Panic Contagion Ratio', fontsize=14)
axes[1,0].set_title('Supply Chain Bottlenecks', fontsize=14)
axes[1,1].set_title('Total Portfolio Value ($)', fontsize=14)

for ax in axes.flat:
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('VORq Agent-Based Crisis Simulation', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('abm_crisis_simulation.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Export Crisis Propagation Matrices

In [ ]:
# Compute propagation characteristics for each severity level
propagation_matrices = {}

for severity_name, history in results.items():
    peak_panic = max(h['panic_ratio'] for h in history)
    time_to_peak = next(i for i, h in enumerate(history) if h['panic_ratio'] == peak_panic)
    initial_value = history[0]['total_portfolio_value']
    final_value = history[-1]['total_portfolio_value']
    max_drawdown = 1 - min(h['total_portfolio_value'] for h in history) / initial_value
    recovery_step = None
    for i, h in enumerate(history):
        if i > time_to_peak and h['panic_ratio'] < 0.05:
            recovery_step = i
            break
    
    propagation_matrices[severity_name] = {
        'peak_panic_ratio': round(peak_panic, 4),
        'time_to_peak_steps': time_to_peak,
        'max_drawdown_pct': round(max_drawdown * 100, 2),
        'portfolio_loss_pct': round((1 - final_value/initial_value) * 100, 2),
        'recovery_step': recovery_step,
        'contagion_multiplier': round(peak_panic / shock_levels[severity_name], 3),
    }

with open('abm_propagation_matrices.json', 'w') as f:
    json.dump(propagation_matrices, f, indent=2)

print('✅ Exported abm_propagation_matrices.json')
for name, data in propagation_matrices.items():
    print(f'\n{name.upper()} crisis:')
    for k, v in data.items():
        print(f'  {k}: {v}')